### **Zero Shot Evaluation**

In [1]:
"""
=============================================================================
  Zero-Shot Evaluation of StarCoder on APPS Dataset (Google Drive)
  No fine-tuning — inference only → compute Pass@k and other metrics
=============================================================================
  Run each section as a separate Colab cell.
=============================================================================
"""

!pip install -q transformers accelerate bitsandbytes sentencepiece protobuf tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 39.7 MB/s eta 0:00:00


In [2]:
!pip install -q numpy pandas matplotlib seaborn

In [3]:
!pip install -q torch torchvision

In [4]:
!pip install -q vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.3/432.3 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/267.7 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 131.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 106.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.8/481.8 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.

### **Configuration Initial**

In [6]:
import os, json, re, subprocess, sys, tempfile, time
from dataclasses import dataclass
from typing import List, Dict, Optional
from collections import Counter, defaultdict
import numpy as np
import torch
from tqdm import tqdm

@dataclass
class EvalConfig:
    apps_root: str = "/content/drive/MyDrive/APPS"
    output_dir: str = "/content/eval_outputs"
    split: str = "test"

    model_name: str = "bigcode/starcoder2-3b"
    trust_remote_code: bool = True
    load_in_4bit: bool = False
    dtype: str = "bfloat16"
    use_flash_attn: bool = False
    use_torch_compile: bool = True
    use_bettertransformer: bool = False


    max_new_tokens: int = 512
    max_prompt_length: int = 1024  ### 1536
    temperature: float = 0.7
    top_p: float = 0.95
    num_samples_per_problem: int = 5  ### 10
    batch_size_generation: int = 5  ## num_samples

    timeout_per_test: int = 4   ## 6
    max_problems: Optional[int] = 10
    difficulty_filter: Optional[str] = None
    num_workers_exec: int = 4

cfg = EvalConfig()
os.makedirs(cfg.output_dir, exist_ok=True)

print(f"APPS root     : {cfg.apps_root}")
print(f"Model         : {cfg.model_name}")
print(f"Samples/prob  : {cfg.num_samples_per_problem}")
print(f"Batch gen size: {cfg.batch_size_generation}")
print(f"Split         : {cfg.split}")
print(f"Difficulty    : {cfg.difficulty_filter or 'all'}")
print(f"bf16={cfg.dtype}  FlashAttn={cfg.use_flash_attn}  "
      f"Compile={cfg.use_torch_compile}")

APPS root     : /content/drive/MyDrive/APPS
Model         : bigcode/starcoder2-3b
Samples/prob  : 5
Batch gen size: 5
Split         : test
Difficulty    : all
bf16=bfloat16  FlashAttn=False  Compile=True


### **Load APPS Dataset from Drive**

In [7]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [8]:
import gzip, pickle
# Load
with gzip.open("/content/drive/MyDrive/apps_dataset.pkl.gz", "rb") as f:
    dataset = pickle.load(f)

### **Load the Model (A100 Optimized)**

In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_model_a100(cfg: EvalConfig):
    print(f"Loading {cfg.model_name} in bf16 with FlashAttention-2 ...")

    tokenizer = AutoTokenizer.from_pretrained(
        cfg.model_name, trust_remote_code=cfg.trust_remote_code
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"  # critical for batched generation

    # ── A100: full bf16, flash_attention_2, no quantisation ──────────────
    model_kwargs = dict(
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=cfg.trust_remote_code,
    )

    if cfg.use_flash_attn:
        model_kwargs["attn_implementation"] = "flash_attention_2"
        print(" FlashAttention-2 enabled")

    model = AutoModelForCausalLM.from_pretrained(cfg.model_name, **model_kwargs)
    model.eval()

    # ── torch.compile for faster inference ───────────────────────────────
    if cfg.use_torch_compile:
        try:
            model = torch.compile(model, mode="reduce-overhead")
            print(" torch.compile enabled (reduce-overhead mode)")
        except Exception as e:
            print(f" torch.compile failed ({e}), continuing without it")

    # ── Memory stats ─────────────────────────────────────────────────────
    total_params = sum(p.numel() for p in model.parameters())
    mem_gb = torch.cuda.memory_allocated() / 1e9
    print(f"Model loaded: {total_params/1e9:.2f}B params")
    print(f"   VRAM used: {mem_gb:.1f} GB / "
          f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    return model, tokenizer


model, tokenizer = load_model_a100(cfg)

# ── Warmup pass (torch.compile compiles on first run) ────────────────────
print("Warmup pass ...")
_ = model.generate(
    **tokenizer("def hello():\n", return_tensors="pt").to(model.device),
    max_new_tokens=32, do_sample=False
)
print("   Warmup complete.\n")


Loading bigcode/starcoder2-3b in bf16 with FlashAttention-2 ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/700 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/12.1G [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


 torch.compile enabled (reduce-overhead mode)
Model loaded: 3.03B params
   VRAM used: 6.1 GB / 42.4 GB
Warmup pass ...
   Warmup complete.



### **Batched Generation (A100 Key optimization)**

In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_model_a100(cfg: EvalConfig):
    print(f"Loading {cfg.model_name} in bf16 with FlashAttention-2 ...")

    tokenizer = AutoTokenizer.from_pretrained(
        cfg.model_name, trust_remote_code=cfg.trust_remote_code
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"  # critical for batched generation

    # ── A100: full bf16, flash_attention_2, no quantisation ──────────────
    model_kwargs = dict(
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=cfg.trust_remote_code,
    )

    if cfg.use_flash_attn:
        model_kwargs["attn_implementation"] = "flash_attention_2"
        print(" FlashAttention-2 enabled")

    model = AutoModelForCausalLM.from_pretrained(cfg.model_name, **model_kwargs)
    model.eval()

    # ── torch.compile for faster inference ───────────────────────────────
    if cfg.use_torch_compile:
        try:
            model = torch.compile(model, mode="reduce-overhead")
            print("torch.compile enabled (reduce-overhead mode)")
        except Exception as e:
            print(f" torch.compile failed ({e}), continuing without it")

    # ── Memory stats ─────────────────────────────────────────────────────
    total_params = sum(p.numel() for p in model.parameters())
    mem_gb = torch.cuda.memory_allocated() / 1e9
    print(f"Model loaded: {total_params/1e9:.2f}B params")
    print(f"   VRAM used: {mem_gb:.1f} GB / "
          f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    return model, tokenizer


model, tokenizer = load_model_a100(cfg)

# ── Warmup pass (torch.compile compiles on first run) ────────────────────
print("Warmup pass ...")
_ = model.generate(
    **tokenizer("def hello():\n", return_tensors="pt").to(model.device),
    max_new_tokens=32, do_sample=False
)
print("   Warmup complete.\n")


Loading bigcode/starcoder2-3b in bf16 with FlashAttention-2 ...


Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


torch.compile enabled (reduce-overhead mode)
Model loaded: 3.03B params
   VRAM used: 12.1 GB / 42.4 GB
Warmup pass ...
   Warmup complete.



### **Parallel Code Execution Engine**

In [11]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def execute_code(code: str, test_input: str, expected_output: str,
                 timeout: int = 6) -> Dict:
    with tempfile.NamedTemporaryFile(mode="w", suffix=".py",
                                      delete=False, dir="/tmp") as f:
        f.write(code)
        fpath = f.name

    result = {"passed": False, "error": None, "actual_output": ""}
    try:
        proc = subprocess.run(
            [sys.executable, fpath],
            input=test_input, capture_output=True, text=True, timeout=timeout,
        )
        result["actual_output"] = proc.stdout.strip()
        if proc.returncode != 0:
            result["error"] = f"RuntimeError: {proc.stderr.strip()[:300]}"
        else:
            actual_lines = [l.rstrip() for l in proc.stdout.strip().splitlines()]
            expected_lines = [l.rstrip() for l in expected_output.strip().splitlines()]
            result["passed"] = actual_lines == expected_lines
    except subprocess.TimeoutExpired:
        result["error"] = "TimeoutError"
    except Exception as e:
        result["error"] = str(e)[:300]
    finally:
        try:
            os.unlink(fpath)
        except OSError:
            pass
    return result


def code_compiles(code: str) -> bool:
    try:
        compile(code, "<eval>", "exec")
        return True
    except SyntaxError:
        return False


def evaluate_solution_parallel(code: str, test_cases: Dict,
                               timeout: int = 6,
                               max_workers: int = 4) -> Dict:
    """
    A100 optimisation: run test cases in parallel threads.
    (Execution is I/O-bound on subprocess, so threading helps.)
    """
    result = {
        "compiles": code_compiles(code),
        "tests_passed": 0, "tests_total": 0,
        "pass_ratio": 0.0, "all_passed": False, "errors": [],
    }
    if not result["compiles"]:
        return result

    inputs = test_cases.get("inputs", [])
    outputs = test_cases.get("outputs", [])
    n_tests = min(len(inputs), len(outputs))
    result["tests_total"] = n_tests
    if n_tests == 0:
        return result

    # ── Parallel execution ───────────────────────────────────────────────
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(execute_code, code, inp, exp, timeout): idx
            for idx, (inp, exp) in enumerate(zip(inputs[:n_tests], outputs[:n_tests]))
        }
        for future in as_completed(futures):
            exec_result = future.result()
            if exec_result["passed"]:
                result["tests_passed"] += 1
            elif exec_result["error"]:
                result["errors"].append(exec_result["error"])

    result["pass_ratio"] = result["tests_passed"] / n_tests
    result["all_passed"] = result["tests_passed"] == n_tests
    return result

### **Full Evaluation Loop**

In [12]:
def build_prompt(problem: Dict) -> str:
    prompt = f"# Problem\n{problem['question']}\n\n"
    if problem["starter_code"]:
        prompt += f"# Starter Code\n{problem['starter_code']}\n\n# Complete Solution\n"
    else:
        prompt += "# Solution\n"
    return prompt


@torch.no_grad()
def generate_solutions_batched(model, tokenizer, problem: Dict, cfg: EvalConfig) -> List[str]:
    prompt = build_prompt(problem)
    inputs = tokenizer(
        [prompt] * cfg.batch_size_generation,
        return_tensors="pt",
        truncation=True,
        max_length=cfg.max_prompt_length,
        padding=True,
    ).to(model.device)

    prompt_len = inputs.input_ids.shape[1]

    outputs = model.generate(
        **inputs,
        max_new_tokens=cfg.max_new_tokens,
        temperature=cfg.temperature,
        top_p=cfg.top_p,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
    )

    solutions = []
    for i in range(outputs.shape[0]):
        generated = tokenizer.decode(outputs[i][prompt_len:], skip_special_tokens=True)
        solutions.append(generated)
    return solutions

In [13]:
def extract_code(raw_generation: str) -> str:
    blocks = re.findall(r"```python\s*(.*?)```", raw_generation, re.DOTALL)
    if blocks:
        return blocks[0].strip()
    blocks = re.findall(r"```\s*(.*?)```", raw_generation, re.DOTALL)
    if blocks:
        return blocks[0].strip()
    stop_patterns = ["\n# Explanation", "\n# Test", "\n\"\"\"", "\n'''",
                     "\nif __name__"]
    code = raw_generation
    for pat in stop_patterns:
        idx = code.find(pat)
        if idx > 50:
            code = code[:idx]
    return code.strip()


def run_full_evaluation(model, tokenizer, dataset: List[Dict],
                        cfg: EvalConfig) -> Dict:
    all_results = []
    generation_times = []
    execution_times = []

    for i, problem in enumerate(tqdm(dataset, desc="Evaluating")):
        pid = problem["problem_id"]

        # ── Batched Generation ───────────────────────────────────────────
        t0 = time.time()
        raw_generations = generate_solutions_batched(model, tokenizer, problem, cfg)
        gen_time = time.time() - t0
        generation_times.append(gen_time)

        # ── Extract & Evaluate ───────────────────────────────────────────
        t1 = time.time()
        problem_result = {
            "problem_id": pid,
            "difficulty": problem["difficulty"],
            "n_samples": len(raw_generations),
            "n_correct": 0,
            "sample_results": [],
            "codes": [],
        }

        for raw in raw_generations:
            code = extract_code(raw)
            problem_result["codes"].append(code)
            eval_result = evaluate_solution_parallel(
                code, problem["test_cases"],
                cfg.timeout_per_test, cfg.num_workers_exec
            )
            problem_result["sample_results"].append(eval_result)
            if eval_result["all_passed"]:
                problem_result["n_correct"] += 1

        exec_time = time.time() - t1
        execution_times.append(exec_time)
        all_results.append(problem_result)

        # ── Progress log every 50 ────────────────────────────────────────
        if (i + 1) % 50 == 0:
            running_pass1 = np.mean([
                1 if r["n_correct"] > 0 else 0 for r in all_results
            ])
            avg_gen = np.mean(generation_times[-50:])
            avg_exec = np.mean(execution_times[-50:])
            vram = torch.cuda.memory_allocated() / 1e9
            print(f"   [{i+1}/{len(dataset)}] Pass@1={running_pass1:.3f} | "
                  f"Gen={avg_gen:.1f}s Exec={avg_exec:.1f}s | VRAM={vram:.1f}GB")

    return {
        "config": {
            "model": cfg.model_name,
            "split": cfg.split,
            "n_samples": cfg.num_samples_per_problem,
            "temperature": cfg.temperature,
            "difficulty": cfg.difficulty_filter,
            "num_problems": len(dataset),
            "gpu": "A100-40GB",
            "precision": cfg.dtype,
            "flash_attn": cfg.use_flash_attn,
            "batch_generation": cfg.batch_size_generation,
        },
        "results": all_results,
        "generation_times": generation_times,
        "execution_times": execution_times,
    }


# ── RUN ──────────────────────────────────────────────────────────────────────
dataset_eval = dataset[:cfg.max_problems]
print(f"\nEvaluating: {len(dataset_eval)} problems × "
      f"{cfg.num_samples_per_problem} samples (batched)\n")

eval_output = run_full_evaluation(model, tokenizer, dataset_eval, cfg)

# Save raw results (without full code to save space)
results_path = os.path.join(cfg.output_dir, "raw_results.json")
save_data = {
    "config": eval_output["config"],
    "results": [{k: v for k, v in r.items() if k != "codes"}
                for r in eval_output["results"]],
}
with open(results_path, "w") as f:
    json.dump(save_data, f, indent=2)
print(f"\nRaw results → {results_path}")


Evaluating: 10 problems × 5 samples (batched)



Evaluating: 100%|██████████| 10/10 [03:15<00:00, 19.55s/it]


Raw results → /content/eval_outputs/raw_results.json


In [14]:
# Compute All Metrics
def pass_at_k(n: int, c: int, k: int) -> float:
    """Unbiased estimator of pass@k (Codex/Chen et al. 2021)."""
    if n - c < k:
        return 1.0
    return 1.0 - np.prod(1.0 - k / np.arange(n - c + 1, n + 1))


def cyclomatic_complexity(code: str) -> int:
    keywords = ["if ", "elif ", "for ", "while ", "except ", " and ", " or "]
    return 1 + sum(code.count(kw) for kw in keywords)


def compute_all_metrics(eval_output: Dict) -> Dict:
    """Compute comprehensive metrics from raw evaluation results."""
    results = eval_output["results"]

    # ── 1. Pass@k
    ks = [1]
    n_samples = eval_output["config"]["n_samples"]
    if n_samples >= 5:
        ks.append(5)
    if n_samples >= 10:
        ks.append(10)

    pass_at_k_scores = {}
    for k in ks:
        scores = [
            pass_at_k(r["n_samples"], r["n_correct"], k)
            for r in results if r["n_samples"] >= k
        ]
        pass_at_k_scores[f"pass@{k}"] = float(np.mean(scores)) if scores else 0.0

    # ── 2. Compilation Rate
    total_samples = 0
    compiled_samples = 0
    for r in results:
        for sr in r["sample_results"]:
            total_samples += 1
            if sr["compiles"]:
                compiled_samples += 1
    compilation_rate = compiled_samples / max(total_samples, 1)

    # ── 3. Average Test Pass Ratio ───────────────────────────────────────
    pass_ratios = [
        sr["pass_ratio"]
        for r in results for sr in r["sample_results"]
        if sr["tests_total"] > 0
    ]
    avg_test_pass_ratio = float(np.mean(pass_ratios)) if pass_ratios else 0.0

    # ── 4. Error Distribution
    error_types = Counter()
    for r in results:
        for sr in r["sample_results"]:
            if not sr["compiles"]:
                error_types["SyntaxError"] += 1
            for err in sr.get("errors", []):
                if "Timeout" in err:
                    error_types["Timeout"] += 1
                elif "Runtime" in err:
                    error_types["RuntimeError"] += 1
                else:
                    error_types["Other"] += 1

    # ── 5. Cyclomatic Complexity
    all_codes = [code for r in results for code in r.get("codes", [])]
    cc_values = [cyclomatic_complexity(c) for c in all_codes if c.strip()]
    cc_stats = {}
    if cc_values:
        cc_stats = {
            "mean": float(np.mean(cc_values)),
            "median": float(np.median(cc_values)),
            "std": float(np.std(cc_values)),
            "p90": float(np.percentile(cc_values, 90)),
        }

    # ── 6. Code Length ───────────────────────────────────────────────────
    lengths = [len(c.split()) for c in all_codes if c.strip()]
    length_stats = {}
    if lengths:
        length_stats = {
            "mean_tokens": float(np.mean(lengths)),
            "median_tokens": float(np.median(lengths)),
            "p90_tokens": float(np.percentile(lengths, 90)),
        }

    # ── 7. Per-Difficulty Breakdown ──────────────────────────────────────
    diff_groups = defaultdict(list)
    for r in results:
        diff_groups[r["difficulty"]].append(r)

    difficulty_breakdown = {}
    for diff, group in diff_groups.items():
        scores = [pass_at_k(r["n_samples"], r["n_correct"], 1) for r in group]
        compile_count = sum(
            1 for r in group for sr in r["sample_results"] if sr["compiles"]
        )
        total_count = sum(len(r["sample_results"]) for r in group)
        difficulty_breakdown[diff] = {
            "count": len(group),
            "pass@1": float(np.mean(scores)),
            "compilation_rate": compile_count / max(total_count, 1),
        }

    # ── 8. Generation Speed ──────────────────────────────────────────────
    gen_times = eval_output.get("generation_times", [])
    speed_stats = {}
    if gen_times:
        speed_stats = {
            "mean_sec_per_problem": float(np.mean(gen_times)),
            "total_time_min": float(np.sum(gen_times) / 60),
        }

    # ── Assemble ─────────────────────────────────────────────────────────
    metrics = {
        "pass_at_k": pass_at_k_scores,
        "compilation_rate": compilation_rate,
        "avg_test_pass_ratio": avg_test_pass_ratio,
        "error_distribution": dict(error_types.most_common()),
        "cyclomatic_complexity": cc_stats,
        "code_length": length_stats,
        "difficulty_breakdown": difficulty_breakdown,
        "generation_speed": speed_stats,
        "total_problems": len(results),
        "total_samples": total_samples,
    }

    return metrics


metrics = compute_all_metrics(eval_output)

# Save metrics
metrics_path = os.path.join(cfg.output_dir, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Metrics saved to {metrics_path}")


💾  Metrics saved to /content/eval_outputs/metrics.json
